# Stage 3: PPO Fine-tuning for ViT5 Medical Summarization

Notebook này tiếp tục huấn luyện checkpoint Stage 2 (SFT) bằng PPO cho bài toán tóm tắt văn bản y tế tiếng Việt. Reward được thiết kế theo các tiêu chí: giữ số liệu quan trọng, kiểm soát độ dài, giảm lặp từ và phạt hallucination số liệu.

**Đầu vào chính**
- Checkpoint Stage 2: `STAGE2_BEST_DIR`
- Dataset gốc: `DATA_ROOT/{train,val,test}/dataset.jsonl`

**Đầu ra chính**
- Best model Stage 3: `BASE_DIR_S3/runs/ppo_run/best_model`
- Log huấn luyện: `ppo_training_log.json`
- Kết quả đánh giá: `eval_val_ppo.json`, `final_results_stage3.json`, `stage3_predictions.jsonl`, `stage3_metrics.csv`

**Thứ tự chạy**
1. Environment setup
2. Imports
3. Configuration
4. Mount Drive and validate paths
5. Load dataset
6. Repair Stage 2 checkpoint if needed
7. Initialize PPO actor/reference models
8. Define reward functions
9. Train with PPO
10. Evaluate on validation set
11. Evaluate once on test set and export results


## 1. Environment Setup

Cell này kiểm tra phiên bản thư viện theo cấu hình đã dùng cho thí nghiệm. Nếu chạy trên runtime mới và thiếu package, đặt `INSTALL_DEPS = True`, chạy cell, sau đó restart runtime trước khi chạy tiếp.


In [ ]:
# # Cell 1 - Environment setup
# # Set INSTALL_DEPS=True only when the runtime is missing the required package versions.
# # After installation, restart the runtime before continuing with Cell 2.

# import sys
# import subprocess
# import importlib.metadata as md

# INSTALL_DEPS = True

# PACKAGES = [
#     "transformers==4.46.3",
#     "tokenizers==0.20.3",
#     "datasets==3.1.0",
#     "evaluate==0.4.3",
#     "accelerate==1.1.1",
#     "sentencepiece==0.2.0",
#     "trl==0.11.4",
#     "rouge-score==0.1.2",
#     "bert-score==0.3.13",
# ]

# mismatched = []
# for spec in PACKAGES:
#     package_name, target_version = spec.split("==")
#     try:
#         current_version = md.version(package_name)
#     except md.PackageNotFoundError:
#         mismatched.append((package_name, "missing", target_version))
#         continue

#     if current_version != target_version:
#         mismatched.append((package_name, current_version, target_version))

# if not mismatched:
#     print("Environment OK: all required package versions are installed.")
# else:
#     print("Packages missing or version-mismatched:")
#     for package_name, current_version, target_version in mismatched:
#         print(f"- {package_name}: {current_version} -> {target_version}")

#     if INSTALL_DEPS:
#         subprocess.check_call([
#             sys.executable,
#             "-m",
#             "pip",
#             "install",
#             "-q",
#             "--no-cache-dir",
#             "--upgrade",
#             *PACKAGES,
#         ])
#         raise RuntimeError(
#             "Installation finished. Restart the runtime, then run the notebook again from Cell 2."
#         )

#     print("Set INSTALL_DEPS=True and rerun this cell if installation is needed.")


Packages missing or version-mismatched:
- transformers: 5.0.0 -> 4.46.3
- tokenizers: 0.22.2 -> 0.20.3
- datasets: 4.0.0 -> 3.1.0
- evaluate: missing -> 0.4.3
- accelerate: 1.13.0 -> 1.1.1
- sentencepiece: 0.2.1 -> 0.2.0
- trl: missing -> 0.11.4
- rouge-score: missing -> 0.1.2
- bert-score: missing -> 0.3.13


RuntimeError: Installation finished. Restart the runtime, then run the notebook again from Cell 2.

## 2. Imports

Import thư viện, metric và các lớp PPO cần dùng.


In [ ]:
# Cell 2 - Imports
# Core utilities, metrics, Transformers and TRL classes.

import os
import re
import gc
import json
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch

# HuggingFace
import evaluate
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from bert_score import score as bertscore

# TRL — PPO cho seq2seq
# AutoModelForSeq2SeqLMWithValueHead them mot value head len tren SFT model
# de PPOTrainer tinh duoc advantage va cap nhat policy
from trl import PPOConfig, PPOTrainer
from trl.models import AutoModelForSeq2SeqLMWithValueHead

# Patch HfFolder de tranh loi token khi dung evaluate offline
import huggingface_hub.hf_api
class _MockHfFolder:
    @classmethod
    def get_token(cls):
        return None
huggingface_hub.hf_api.HfFolder = _MockHfFolder

print("Import thanh cong.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Import thanh cong.
PyTorch: 2.10.0+cu128
CUDA available: True


## 3. Configuration

Khai báo đường dẫn, hyperparameter PPO, thiết lập reward, precision và thư mục output.


In [ ]:
# Cell 3 - Global configuration
# Update these three paths before running on a new Drive/runtime.

# Paths

STAGE2_BEST_DIR = (
    "/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP"
    "/ViT5_stage_2/runs/vit5_base_stage2/checkpoints/checkpoint-735"
)

DATA_ROOT = (
    "/content/drive/MyDrive/MyNLPData/data"
)

BASE_DIR_S3 = (
    "/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3"
)

# ------------------------------------------------------------------
# HYPERPARAMETER PPO
# Conservative: learning rate thap, ppo_epochs it, KL coef cao de bao ve SFT.
# Voi dataset 780 mau, day la thiet lap an toan nhat.
# ------------------------------------------------------------------

CONFIG = {
    # Nhan dien
    "MODEL_LABEL": "ViT5-base Stage2 -> Stage3 PPO",
    "STAGE2_BEST_DIR": STAGE2_BEST_DIR,
    "DATA_ROOT": DATA_ROOT,
    "BASE_DIR_S3": BASE_DIR_S3,

    # Du lieu cho RL
    # Chi dung 780 dong goc, khong dung augmented (tranh reward bias do
    # noi dung augmented giong ban goc).
    "USE_AUGMENTED_FOR_RL": False,

    # Loc augmented chat luong cao neu muon mo rong dataset
    "USE_QUALITY_AUGMENTED": False,
    "AUGMENTED_MIN_RATIO": 0.18,
    "AUGMENTED_MAX_RATIO": 0.27,
    "AUGMENTED_MAX_SIMILARITY": 0.85,

    # Tokenizer / generation (cho eval, dung beam search)
    "MAX_INPUT_LEN":   1024,
    "MAX_TARGET_LEN":   512,
    "GEN_MAX_LEN":      320,
    "GEN_MIN_LEN":       64,
    "NUM_BEAMS":          4,
    "NO_REPEAT_NGRAM":    3,
    "LENGTH_PENALTY":   1.0,

    # ------------------------------------------------------------------
    # PPO hyperparameter
    # ------------------------------------------------------------------
    # learning_rate: thap hon SFT (3e-5) nhieu lan, bat dau 1e-6
    "PPO_LR":            1e-6,
    # ppo_epochs: so lan update tren moi batch
    "PPO_EPOCHS":           1,
    # batch_size co dinh trong TRL — drop incomplete batch cuoi epoch
    "PPO_BATCH_SIZE":      16,
    # mini_batch_size: chia batch lon thanh mini-batch de giam VRAM
    "PPO_MINI_BATCH":       8,
    # so epoch RL tren toan bo dataset
    "RL_EPOCHS":            3,
    # KL penalty coefficient — khoi dau cao 0.3 de giu model gan SFT
    "KL_COEF":            0.3,
    # PPO clip range
    "CLIP_RANGE":         0.2,

    # ------------------------------------------------------------------
    # Sampling cho training
    # KHONG nen dung top_k/top_p truncation voi PPO seq2seq vi sampled
    # log_prob khac voi distribution truncated -> KL estimate co the bi am.
    # ------------------------------------------------------------------
    "TRAIN_TOP_K":        0,
    "TRAIN_TOP_P":      1.0,
    "TRAIN_TEMPERATURE": 1.0,

    # KL penalty mode trong TRL: "abs" tranh KL am cong nguoc vao reward
    "KL_PENALTY_MODE":  "abs",

    # KL detection: voi KL_PENALTY_MODE="abs", objective/kl trong TRL stats
    # luon >= 0 (da abs), nen check "KL am" vo nghia. Thay bang check KL
    # qua cao -> dau hieu policy diverge khoi SFT.
    "KL_HIGH_THRESHOLD":  5.0,
    "KL_HIGH_STREAK":       3,

    # ------------------------------------------------------------------
    # Reward weights — tai su dung tu Stage 2
    # ------------------------------------------------------------------
    "W_NUMBER":         0.45,
    "W_LENGTH":         0.20,
    "W_HALLUCINATION":  0.20,
    "W_DIVERSITY":      0.15,

    # Hard penalty cho hallucination cao
    "HALLUC_HARD_THRESHOLD": 0.5,
    "HALLUC_HARD_PENALTY":   1.0,

    # Length constraints
    # Avg gold ratio thuc te ~0.235; range hep (0.20-0.25) khien nhieu gold
    # cung "khong tuan thu" -> reward bias. Noi rong [0.18, 0.27] khop voi
    # phan phoi gold.
    "MIN_RATIO":   0.18,
    "MAX_RATIO":   0.27,
    "TARGET_RATIO": 0.225,

    # Number reward — dung target tu gold de tranh ep model copy moi so
    # trong source. Khong co so target -> reward 0.0 (khong cong free).
    "NUMBER_USE_GOLD_FOR_REWARD": True,
    "NUMBER_NO_TARGET_REWARD":    0.0,

    # ------------------------------------------------------------------
    # Eval & checkpoint
    # ------------------------------------------------------------------
    "EVAL_BATCH":           8,
    "EVAL_EVERY_N_STEPS":  25,    # eval tren val subset moi N step
    "EVAL_VAL_SUBSET":     30,    # so mau val cho quick eval

    # Best-model selection: composite cua rougeL va reward (cung scale 0-100)
    "BEST_W_ROUGEL":  0.5,
    "BEST_W_REWARD":  0.5,
}

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

# Precision
# - BF16: an toan cho PPO neu GPU support (A100, H100, T4 gen moi).
# - FP16 BI LOAI: PPO + value head + gradient scaling khong on dinh.
# - FP32: fallback an toan.
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32

# Thu muc output
OUT_DIR     = Path(BASE_DIR_S3) / "runs" / "ppo_run"
CKPT_DIR    = OUT_DIR / "checkpoints"
BEST_DIR_S3 = OUT_DIR / "best_model"
RESULTS_DIR = OUT_DIR / "results"
# Tao thu muc output sau khi Drive da mount trong Cell 4.
# Neu tao /content/drive truoc khi mount, Colab co the de lai mount point loi.

print("Model:", CONFIG["MODEL_LABEL"])
print("Stage 2 ckpt:", CONFIG["STAGE2_BEST_DIR"])
print("Data root  :", CONFIG["DATA_ROOT"])
print("Output dir :", OUT_DIR)
print("Device     :", DEVICE)
print("Precision  :", "bf16" if USE_BF16 else "fp32")
CONFIG


Model: ViT5-base Stage2 -> Stage3 PPO
Stage 2 ckpt: /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2/runs/vit5_base_stage2/checkpoints/checkpoint-735
Data root  : /content/drive/MyDrive/MyNLPData/data
Output dir : /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run
Device     : cuda
Precision  : bf16


{'MODEL_LABEL': 'ViT5-base Stage2 -> Stage3 PPO',
 'STAGE2_BEST_DIR': '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2/runs/vit5_base_stage2/checkpoints/checkpoint-735',
 'DATA_ROOT': '/content/drive/MyDrive/MyNLPData/data',
 'BASE_DIR_S3': '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3',
 'USE_AUGMENTED_FOR_RL': False,
 'USE_QUALITY_AUGMENTED': False,
 'AUGMENTED_MIN_RATIO': 0.18,
 'AUGMENTED_MAX_RATIO': 0.27,
 'AUGMENTED_MAX_SIMILARITY': 0.85,
 'MAX_INPUT_LEN': 1024,
 'MAX_TARGET_LEN': 512,
 'GEN_MAX_LEN': 320,
 'GEN_MIN_LEN': 64,
 'NUM_BEAMS': 4,
 'NO_REPEAT_NGRAM': 3,
 'LENGTH_PENALTY': 1.0,
 'PPO_LR': 1e-06,
 'PPO_EPOCHS': 1,
 'PPO_BATCH_SIZE': 16,
 'PPO_MINI_BATCH': 8,
 'RL_EPOCHS': 3,
 'KL_COEF': 0.3,
 'CLIP_RANGE': 0.2,
 'TRAIN_TOP_K': 0,
 'TRAIN_TOP_P': 1.0,
 'TRAIN_TEMPERATURE': 1.0,
 'KL_PENALTY_MODE': 'abs',
 'KL_HIGH_THRESHOLD': 5.0,
 'KL_HIGH_STREAK': 3,
 'W_NUMBER': 0.45,
 'W_LENGTH': 0.2,
 'W_HALLUCINATION': 0.2,
 'W_DIVER

## 4. Mount Drive and Validate Paths

Mount Google Drive, kiểm tra checkpoint Stage 2 và dataset train/val/test.


In [ ]:
# Cell 3B - Khong xoa /content/drive bang rm -rf.
# Cell 4 se flush/unmount va mount lai Drive an toan.
print("Skip xoa /content/drive; hay chay Cell 4 de remount Drive.")

Skip xoa /content/drive; hay chay Cell 4 de remount Drive.


In [ ]:
# Cell 4 - Mount Drive and validate required files.

from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception as e:
    print("Bo qua flush/unmount cu:", repr(e))

drive.mount("/content/drive", force_remount=True)

# Ep dong bo danh muc file
!ls -d "/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/"*

# Kiem tra Stage 2 checkpoint ton tai
stage2_path = Path(CONFIG["STAGE2_BEST_DIR"])
if not stage2_path.exists():
    raise FileNotFoundError(
        f"Khong tim thay Stage 2 checkpoint tai: {stage2_path}\n"
        "Kiem tra lai STAGE2_BEST_DIR trong Cell 3."
    )

# Kiem tra data
data_path = Path(CONFIG["DATA_ROOT"])
for split in ("train", "val", "test"):
    p = data_path / split / "dataset.jsonl"
    if not p.exists():
        raise FileNotFoundError(
            f"Thieu file: {p}\n"
            "Kiem tra lai DATA_ROOT trong Cell 3."
        )

print("Stage 2 checkpoint ton tai:", stage2_path)
print("Data OK: train / val / test")

# Tao va kiem tra quyen ghi output sau khi Drive da mount.
def assert_writable_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    test_file = path / ".write_test.tmp"
    try:
        test_file.write_text("ok", encoding="utf-8")
        test_file.unlink()
    except Exception as e:
        raise PermissionError(
            f"Khong ghi duoc vao thu muc output: {path}. "
            "Hay remount Drive hoac doi BASE_DIR_S3 sang thu muc ban co quyen ghi."
        ) from e

for d in (OUT_DIR, CKPT_DIR, RESULTS_DIR):
    assert_writable_dir(d)
BEST_DIR_S3.mkdir(parents=True, exist_ok=True)

print("Output dirs writable:", OUT_DIR)


Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
'/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_1'
'/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2'
'/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3'
Stage 2 checkpoint ton tai: /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2/runs/vit5_base_stage2/checkpoints/checkpoint-735
Data OK: train / val / test
Output dirs writable: /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run


## 5. Load Dataset

Đọc dữ liệu gốc, chuẩn hóa record và tạo tập train dùng cho PPO.


In [ ]:
# Cell 5 - Load and normalize dataset.
# PPO uses the original dataset by default; high-quality augmented data is optional.

NUM_RE = re.compile(r"\d+(?:[.,]\d+)*")

def extract_numbers(text):
    return set(NUM_RE.findall(text or ""))

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalize_record(row):
    source  = str(row.get("source",  "")).strip()
    summary = str(row.get("summary", "")).strip()
    return {
        "id":           row.get("id", ""),
        "url":          row.get("url", ""),
        "category":     row.get("category", ""),
        "quality_tag":  row.get("quality_tag", ""),
        "source":       source,
        "summary":      summary,
        "src_word_count":  len(source.split()),
        "tgt_word_count":  len(summary.split()),
        "length_ratio":    len(summary.split()) / max(len(source.split()), 1),
    }

def simple_similarity(a, b):
    # Uoc luong do tuong dong bang Jaccard tren tap tu
    # Dung de loc augmented qua giong ban goc, khong can thu vien ngoai
    set_a, set_b = set(a.split()), set(b.split())
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

data_path = Path(CONFIG["DATA_ROOT"])

# Load 780 dong goc
train_raw = [normalize_record(r) for r in load_jsonl(data_path / "train" / "dataset.jsonl")]
val_raw   = [normalize_record(r) for r in load_jsonl(data_path / "val"   / "dataset.jsonl")]
test_raw  = [normalize_record(r) for r in load_jsonl(data_path / "test"  / "dataset.jsonl")]

train_rl = list(train_raw)  # bat dau tu 780 dong goc

# Mo rong voi augmented chat luong cao (tuy chon)
aug_path = data_path / "train" / "train_augmented.jsonl"
if CONFIG["USE_QUALITY_AUGMENTED"] and aug_path.exists():
    aug_raw  = [normalize_record(r) for r in load_jsonl(aug_path)]
    # Lay map id goc -> summary goc de tinh similarity
    orig_map = {r["id"]: r["summary"] for r in train_raw}
    added = 0
    for row in aug_raw:
        if row["id"] in orig_map:
            continue  # bo qua ban goc da co roi
        ratio = row["length_ratio"]
        if not (CONFIG["AUGMENTED_MIN_RATIO"] <= ratio <= CONFIG["AUGMENTED_MAX_RATIO"]):
            continue
        # Lay id goc tu id dang "abc_safe_aug_1"
        orig_id = re.sub(r"_safe_aug_\d+$", "", row["id"])
        orig_sum = orig_map.get(orig_id, "")
        if orig_sum and simple_similarity(row["summary"], orig_sum) > CONFIG["AUGMENTED_MAX_SIMILARITY"]:
            continue
        train_rl.append(row)
        added += 1
    print(f"Mo rong dataset voi {added} augmented mau chat luong cao.")

# Loc bo dong trong
train_rl = [r for r in train_rl if r["source"] and r["summary"]]
random.shuffle(train_rl)

print(f"RL train : {len(train_rl):,} mau")
print(f"Val      : {len(val_raw):,} mau")
print(f"Test     : {len(test_raw):,} mau")
print("Category train:", dict(Counter(r["category"] for r in train_rl)))
print("Avg length ratio train:", round(float(np.mean([r["length_ratio"] for r in train_rl])), 4))


RL train : 780 mau
Val      : 94 mau
Test     : 105 mau
Category train: {'suc-khoe': 223, 'song-khoe': 197, 'cac-benh': 72, 'vaccine': 133, 'tin-tuc': 155}
Avg length ratio train: 0.2352


## 6A. Repair Stage 2 Checkpoint

Nếu checkpoint Stage 2 bị thiếu `config.json` hoặc tokenizer files, tạo một bản load riêng cho Stage 3 mà không ghi đè checkpoint gốc.


In [ ]:
# Cell 6A - Repair Stage 2 checkpoint for loading.
# The repaired copy is written under Stage 3 output and never overwrites Stage 2.

from transformers import AutoConfig, T5Tokenizer

# Cell 6a: Repair checkpoint Stage 2 neu bi mat config/tokenizer.
# Khong ghi de STAGE2_BEST_DIR goc. Ta tao mot thu muc load rieng cho Stage 3.

STAGE2_RAW_DIR = Path(CONFIG["STAGE2_BEST_DIR"])
STAGE2_LOAD_DIR = Path(CONFIG["BASE_DIR_S3"]) / "stage2_repaired_for_stage3"

STANDARD_WEIGHT_NAMES = {
    "model.safetensors",
    "pytorch_model.bin",
    "tf_model.h5",
    "flax_model.msgpack",
}


def has_required_hf_files(path):
    path = Path(path)
    if path.is_file():
        return False
    has_config = (path / "config.json").exists()
    has_tokenizer = any(
        (path / name).exists()
        for name in ("tokenizer_config.json", "spiece.model", "tokenizer.json")
    )
    has_weight = any((path / name).exists() for name in STANDARD_WEIGHT_NAMES)
    return has_config and has_tokenizer and has_weight


def find_stage2_weight_files(path):
    path = Path(path)
    if path.is_file():
        return [path]
    exact = [path / name for name in STANDARD_WEIGHT_NAMES if (path / name).exists()]
    if exact:
        return exact

    # Fallback cho truong hop checkpoint chi con mot file weight bi doi ten.
    candidates = []
    for pattern in ("*.safetensors", "*.bin", "*.pt", "*.pth", "*.tensors"):
        candidates.extend(path.glob(pattern))
    candidates = [x for x in candidates if x.is_file()]
    if not candidates:
        raise FileNotFoundError(
            f"Khong thay weight file trong {path}. Can co model.safetensors/pytorch_model.bin "
            "hoac mot file .safetensors/.bin/.pt/.pth/.tensors."
        )
    return [max(candidates, key=lambda x: x.stat().st_size)]


def infer_standard_weight_name(src):
    lower = Path(src).name.lower()
    if lower == "model.safetensors" or lower.endswith(".safetensors"):
        return "model.safetensors"
    if lower == "pytorch_model.bin" or lower.endswith((".bin", ".pt", ".pth")):
        return "pytorch_model.bin"
    if lower.endswith(".tensors"):
        # Ten .tensors khong phai chuan HF. Thu nhan dien safetensors truoc,
        # neu khong doc duoc thi coi nhu torch state_dict.
        try:
            from safetensors import safe_open
            with safe_open(str(src), framework="pt", device="cpu") as f:
                _ = list(f.keys())[:1]
            return "model.safetensors"
        except Exception:
            return "pytorch_model.bin"
    return Path(src).name


def repair_stage2_checkpoint(raw_dir, load_dir):
    raw_dir = Path(raw_dir)
    load_dir = Path(load_dir)

    if has_required_hf_files(raw_dir):
        print("Stage 2 checkpoint da du file HF, dung truc tiep:", raw_dir)
        return raw_dir

    load_dir.mkdir(parents=True, exist_ok=True)

    # Recreate config/tokenizer tu base model goc. Chi dung cach nay khi Stage 2 khong them special tokens.
    cfg = AutoConfig.from_pretrained("VietAI/vit5-base")
    cfg.save_pretrained(load_dir)

    tok = T5Tokenizer.from_pretrained("VietAI/vit5-base", legacy=True)
    tok.save_pretrained(load_dir)

    copied = []
    for src in find_stage2_weight_files(raw_dir):
        dst_name = infer_standard_weight_name(src)
        dst = load_dir / dst_name
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
        copied.append(dst.name)

    print("Stage 2 checkpoint thieu config/tokenizer -> da tao ban repair rieng:", load_dir)
    print("Weight files:", copied)
    return load_dir


STAGE2_LOAD_DIR = repair_stage2_checkpoint(STAGE2_RAW_DIR, STAGE2_LOAD_DIR)
CONFIG["STAGE2_LOAD_DIR"] = str(STAGE2_LOAD_DIR)
print("Stage 3 se load model tu:", CONFIG["STAGE2_LOAD_DIR"])




Stage 2 checkpoint da du file HF, dung truc tiep: /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2/runs/vit5_base_stage2/checkpoints/checkpoint-735
Stage 3 se load model tu: /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2/runs/vit5_base_stage2/checkpoints/checkpoint-735


## 6B. Initialize Actor and Reference Models

Load actor model có value head để PPO cập nhật, đồng thời tạo reference model đóng băng để tính KL.


In [ ]:
# Cell 6B - Initialize actor and reference models.
# Actor is trainable; reference model is frozen and used for KL control.

def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


clear_cuda()

stage2_load_dir = CONFIG.get("STAGE2_LOAD_DIR", CONFIG["STAGE2_BEST_DIR"])

# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained(
    stage2_load_dir,
    legacy=True,
)
print(f"Tokenizer: {type(tokenizer).__name__}, vocab={tokenizer.vocab_size}")

# Actor model — se duoc cap nhat
actor_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(
    stage2_load_dir,
    torch_dtype=MODEL_DTYPE,
)
actor_model = actor_model.to(device=DEVICE, dtype=MODEL_DTYPE)

# Ref model — dong bang
ref_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(
    stage2_load_dir,
    torch_dtype=MODEL_DTYPE,
)
ref_model = ref_model.to(device=DEVICE, dtype=MODEL_DTYPE)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

n_params = sum(p.numel() for p in actor_model.parameters()) / 1e6
print(f"Actor model: {n_params:.1f}M params, dtype={MODEL_DTYPE}")
print(f"Ref model  : dong bang, dtype={MODEL_DTYPE}")

# Load metric
rouge = evaluate.load("rouge")

print("Khoi tao xong.")


Tokenizer: T5Tokenizer, vocab=36000
Actor model: 226.0M params, dtype=torch.bfloat16
Ref model  : dong bang, dtype=torch.bfloat16


Khoi tao xong.


## 7. Reward Functions

Reward kết hợp number recall, length score, diversity và number hallucination penalty. Number recall ưu tiên số liệu xuất hiện trong gold summary để tránh ép model copy mọi số trong source.


In [ ]:
# Cell 7 - Reward functions.
# The breakdown helper returns component scores for logging and debugging.

def number_recall(src, pred, target_text=None):
    """Ti le so lieu can giu xuat hien trong pred.

    target_text != None -> target la so trong gold summary.
    target_text  = None -> target la so trong source.
    Khong co so target  -> NUMBER_NO_TARGET_REWARD (mac dinh 0.0).
    """
    target_nums = (
        extract_numbers(target_text) if target_text is not None
        else extract_numbers(src)
    )
    if not target_nums:
        return float(CONFIG.get("NUMBER_NO_TARGET_REWARD", 0.0))
    return len(target_nums & extract_numbers(pred)) / len(target_nums)


def number_hallucination(src, pred):
    """Ti le so lieu trong pred KHONG co trong src."""
    pred_nums = extract_numbers(pred)
    if not pred_nums:
        return 0.0
    return len(pred_nums - extract_numbers(src)) / len(pred_nums)


def length_ratio_fn(src, pred):
    return len(str(pred).split()) / max(len(str(src).split()), 1)


def length_score(src, pred):
    """1.0 neu ratio nam trong [MIN_RATIO, MAX_RATIO], giam dan neu lech."""
    ratio = length_ratio_fn(src, pred)
    if CONFIG["MIN_RATIO"] <= ratio <= CONFIG["MAX_RATIO"]:
        return 1.0
    return max(0.0, 1.0 - abs(ratio - CONFIG["TARGET_RATIO"]) / CONFIG["TARGET_RATIO"])


def diversity_score(pred):
    """Ti le tu doc nhat trong pred. Thap = lap tu = reward hack."""
    words = str(pred).split()
    if not words:
        return 0.0
    return len(set(words)) / len(words)


def compute_reward_breakdown(src, pred, gold=None):
    """Tinh tung component reward + reward scalar tong (clip [-1, 1]).

    Return dict gom: reward, number_recall, hallucination, length_score,
    diversity, hard_penalty.
    """
    target_for_numbers = (
        gold if CONFIG.get("NUMBER_USE_GOLD_FOR_REWARD", True) else None
    )

    nr = number_recall(src, pred, target_for_numbers)
    nh = number_hallucination(src, pred)
    ls = length_score(src, pred)
    ds = diversity_score(pred)

    raw = (
        CONFIG["W_NUMBER"]      * nr
        + CONFIG["W_LENGTH"]    * ls
        + CONFIG["W_DIVERSITY"] * ds
        - CONFIG["W_HALLUCINATION"] * nh
    )

    hard = 0.0
    if nh > CONFIG["HALLUC_HARD_THRESHOLD"]:
        hard = -CONFIG["HALLUC_HARD_PENALTY"]
        raw += hard

    reward = float(np.clip(raw, -1.0, 1.0))
    return {
        "reward":        reward,
        "number_recall": float(nr),
        "hallucination": float(nh),
        "length_score":  float(ls),
        "diversity":     float(ds),
        "hard_penalty":  float(hard),
    }


def compute_reward(src, pred, gold=None):
    return compute_reward_breakdown(src, pred, gold)["reward"]


# Kiem tra nhanh
test_src_ex  = "Bệnh nhân bị cao huyết áp 120/80, nhịp tim 90, sốt 38 độ."
test_gold_ex = "Bệnh nhân cao huyết áp 120/80, nhịp tim 90."
test_pred_ex = "Bệnh nhân bị cao huyết áp 120/80, nhịp tim 90."
no_num_src   = "Bệnh nhân được khuyến nghị nghỉ ngơi và uống đủ nước."
no_num_pred  = "Bệnh nhân nên nghỉ ngơi và uống đủ nước."

print("Co so:    ", compute_reward_breakdown(test_src_ex, test_pred_ex, test_gold_ex))
print("Khong so: ", compute_reward_breakdown(no_num_src, no_num_pred, no_num_src))
print("Reward functions da san sang.")


Co so:     {'reward': 0.6, 'number_recall': 1.0, 'hallucination': 0.0, 'length_score': 0.0, 'diversity': 1.0, 'hard_penalty': 0.0}
Khong so:  {'reward': 0.15, 'number_recall': 0.0, 'hallucination': 0.0, 'length_score': 0.0, 'diversity': 1.0, 'hard_penalty': 0.0}
Reward functions da san sang.


## 8. PPO Training

Sinh response bằng `ppo_trainer.generate`, tính reward một lần cho mỗi batch, cập nhật policy bằng PPO và lưu best checkpoint theo composite metric trên validation subset.


In [ ]:
# Cell 8 - PPO training loop.
# Generate responses through PPOTrainer, compute reward once, and save the best validation checkpoint.

from tqdm.auto import tqdm

stage2_load_dir = CONFIG.get("STAGE2_LOAD_DIR", CONFIG["STAGE2_BEST_DIR"])

ppo_config = PPOConfig(
    model_name=stage2_load_dir,
    learning_rate=CONFIG["PPO_LR"],
    batch_size=CONFIG["PPO_BATCH_SIZE"],
    mini_batch_size=CONFIG["PPO_MINI_BATCH"],
    ppo_epochs=CONFIG["PPO_EPOCHS"],
    init_kl_coef=CONFIG["KL_COEF"],
    kl_penalty=CONFIG["KL_PENALTY_MODE"],
    cliprange=CONFIG["CLIP_RANGE"],
    cliprange_value=CONFIG["CLIP_RANGE"],
    adap_kl_ctrl=False,
    seed=SEED,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=actor_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
)

# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------

def normalize_for_rouge(text):
    return "\n".join(str(text).strip().split())


def tensor_to_float(value):
    if isinstance(value, torch.Tensor):
        return float(value.detach().cpu().item())
    if hasattr(value, "item"):
        return float(value.item())
    return float(value)


def fixed_eval_subset(rows, n_samples=30):
    """Lay subset val co seed, tranh bias do lay 30 dong dau file."""
    if n_samples is None or n_samples >= len(rows):
        return rows
    rng = random.Random(SEED)
    indices = list(range(len(rows)))
    rng.shuffle(indices)
    chosen = sorted(indices[:n_samples])
    return [rows[i] for i in chosen]


def quick_eval_on_val(n_samples=30):
    """Sinh nhanh tren n_samples mau val (beam search), tra ve dict.

    Tra ve {"rougeL": float, "reward_mean": float} de Cell 8 build composite
    metric tu val reward thay vi reward sampled tu train batch (noisy).

    Dung try/finally de bao dam model.train() duoc restore ngay ca khi
    exception (OOM giua beam search, KeyboardInterrupt). Neu khong, batch
    PPO tiep theo se chay voi dropout off -> log_prob mismatch (cung pattern
    voi KL am cu).
    """
    was_training = actor_model.training
    actor_model.eval()
    try:
        subset = fixed_eval_subset(val_raw, n_samples=n_samples)
        srcs  = [r["source"]  for r in subset]
        golds = [r["summary"] for r in subset]
        preds = []

        for i in range(0, len(srcs), CONFIG["EVAL_BATCH"]):
            batch_texts = srcs[i:i + CONFIG["EVAL_BATCH"]]
            enc = tokenizer(
                batch_texts,
                max_length=CONFIG["MAX_INPUT_LEN"],
                truncation=True,
                padding=True,
                return_tensors="pt",
            ).to(DEVICE)

            with torch.no_grad():
                ids = actor_model.generate(
                    **enc,
                    max_new_tokens=CONFIG["GEN_MAX_LEN"],
                    min_new_tokens=CONFIG["GEN_MIN_LEN"],
                    num_beams=CONFIG["NUM_BEAMS"],
                    no_repeat_ngram_size=CONFIG["NO_REPEAT_NGRAM"],
                    length_penalty=CONFIG["LENGTH_PENALTY"],
                    early_stopping=True,
                )
            preds.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))

        result = rouge.compute(
            predictions=[normalize_for_rouge(x) for x in preds],
            references=[normalize_for_rouge(x) for x in golds],
            use_stemmer=False,
        )
        val_rewards = [
            compute_reward(s, p, g)
            for s, p, g in zip(srcs, preds, golds)
        ]
        return {
            "rougeL":      round(result["rougeL"] * 100, 4),
            "reward_mean": round(float(np.mean(val_rewards)), 4),
        }
    finally:
        if was_training:
            actor_model.train()


def save_actor(out_dir, step=None):
    """Save chi base model + tokenizer, KHONG save value head.

    Cell 9/10 load lai bang AutoModelForSeq2SeqLM (khong co v_head) nen
    save kieu nay tranh warning unexpected keys. Neu Google Drive bi loi
    permission khi overwrite best_model/config.json, fallback sang thu muc
    best_model_step_XXXX de tranh ghi de file bi lock.
    """
    out_dir = Path(out_dir)
    candidates = [out_dir]
    if step is not None:
        candidates.append(out_dir.parent / f"{out_dir.name}_step_{int(step):04d}")

    last_error = None
    for target_dir in candidates:
        try:
            target_dir.mkdir(parents=True, exist_ok=True)
            actor_model.pretrained_model.save_pretrained(str(target_dir))
            tokenizer.save_pretrained(str(target_dir))
            return target_dir
        except PermissionError as e:
            last_error = e
            print(f"[WARN] Khong ghi duoc {target_dir}: {e}")

    raise PermissionError(
        "Khong the luu model vao Drive. Hay chay lai Cell 4 de remount "
        "Drive hoac doi BASE_DIR_S3 sang thu muc ban co quyen ghi."
    ) from last_error


generation_kwargs = dict(
    do_sample=True,
    top_k=CONFIG["TRAIN_TOP_K"],
    top_p=CONFIG["TRAIN_TOP_P"],
    temperature=CONFIG["TRAIN_TEMPERATURE"],
    max_new_tokens=CONFIG["GEN_MAX_LEN"],
    min_new_tokens=CONFIG["GEN_MIN_LEN"],
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    no_repeat_ngram_size=CONFIG["NO_REPEAT_NGRAM"],
)

# ------------------------------------------------------------------
# Training loop
# ------------------------------------------------------------------

training_log     = []
best_score       = -float("inf")
best_meta        = {}
global_step      = 0
bad_kl_streak    = 0
skipped_batches  = 0
stats_keys_logged = False

batches_per_epoch = len(train_rl) // CONFIG["PPO_BATCH_SIZE"]
print(
    f"Bat dau PPO Training - "
    f"{CONFIG['RL_EPOCHS']} RL epoch x {batches_per_epoch} batch/epoch, "
    f"batch={CONFIG['PPO_BATCH_SIZE']}, "
    f"kl_penalty={CONFIG['KL_PENALTY_MODE']}, "
    f"dtype={MODEL_DTYPE}"
)
print("-" * 80)

for rl_epoch in range(CONFIG["RL_EPOCHS"]):
    random.shuffle(train_rl)

    pbar = tqdm(
        range(0, len(train_rl), CONFIG["PPO_BATCH_SIZE"]),
        desc=f"RL Epoch {rl_epoch+1}",
        leave=True,
    )

    for batch_start in pbar:
        batch = train_rl[batch_start : batch_start + CONFIG["PPO_BATCH_SIZE"]]

        # PPOConfig.batch_size co dinh — drop incomplete batch cuoi epoch.
        if len(batch) != CONFIG["PPO_BATCH_SIZE"]:
            skipped_batches += 1
            continue

        sources = [r["source"]  for r in batch]
        golds   = [r["summary"] for r in batch]

        # Tokenize
        query_tensors = []
        for src in sources:
            enc = tokenizer(
                src,
                max_length=CONFIG["MAX_INPUT_LEN"],
                truncation=True,
                return_tensors="pt",
            )
            query_tensors.append(enc["input_ids"].squeeze(0).to(DEVICE))

        # Generate — KHONG eval()/train() de tranh dropout mismatch.
        with torch.no_grad():
            response_tensors = ppo_trainer.generate(
                query_tensors,
                batch_size=CONFIG["PPO_MINI_BATCH"],
                **generation_kwargs,
            )

        predictions = tokenizer.batch_decode(
            response_tensors,
            skip_special_tokens=True,
        )

        # Tinh reward 1 lan, lay ca breakdown de log
        breakdowns = [
            compute_reward_breakdown(s, p, g)
            for s, p, g in zip(sources, predictions, golds)
        ]
        raw_rewards = [b["reward"] for b in breakdowns]
        rewards = [
            torch.tensor(float(r), device=DEVICE)
            for r in raw_rewards
        ]

        # PPO step
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

        # Verify stats keys mot lan dau, sau do strict access (raise neu key
        # bi rename — tranh silent fallback 0.0 lam KL detection vo dung).
        if not stats_keys_logged:
            present_keys = sorted(stats.keys())
            print(f"\n[INFO] PPO stats keys: {present_keys}")
            for required in ("objective/kl", "ppo/loss/total"):
                if required not in stats:
                    raise KeyError(
                        f"PPO stats thieu key '{required}'. "
                        f"Co the do TRL doi key. Keys hien co: {present_keys}"
                    )
            stats_keys_logged = True

        mean_reward = float(np.mean(raw_rewards))
        kl_val      = tensor_to_float(stats["objective/kl"])
        ppo_loss    = tensor_to_float(stats["ppo/loss/total"])

        # KL detection: voi KL_PENALTY_MODE="abs", kl_val luon >= 0 (TRL da
        # abs). Thay vi check KL am (vo nghia trong abs mode), check KL qua
        # cao -> dau hieu policy diverge khoi SFT.
        if kl_val > CONFIG["KL_HIGH_THRESHOLD"]:
            bad_kl_streak += 1
            print(
                f"\n[WARN] KL cao bat thuong step {global_step}: {kl_val:.4f} "
                f"(streak={bad_kl_streak}/{CONFIG['KL_HIGH_STREAK']})"
            )
        else:
            bad_kl_streak = 0

        if bad_kl_streak >= CONFIG["KL_HIGH_STREAK"]:
            raise RuntimeError(
                f"KL diverge {bad_kl_streak} buoc lien tuc (>"
                f"{CONFIG['KL_HIGH_THRESHOLD']}). Dung training de tranh "
                "checkpoint hong; hay giam PPO_LR / KL_COEF hoac kiem tra "
                "checkpoint Stage 2."
            )

        # Aggregate breakdown cho log
        avg_gen_len = float(np.mean([len(pred.split()) for pred in predictions]))
        agg = {
            "number_recall": float(np.mean([b["number_recall"] for b in breakdowns])),
            "hallucination": float(np.mean([b["hallucination"] for b in breakdowns])),
            "length_score":  float(np.mean([b["length_score"]  for b in breakdowns])),
            "diversity":     float(np.mean([b["diversity"]     for b in breakdowns])),
            "hard_pen_rate": float(np.mean([1.0 if b["hard_penalty"] < 0 else 0.0 for b in breakdowns])),
        }
        recent_rewards = [x["mean_reward"] for x in training_log[-20:]]
        reward_ma20 = float(np.mean(recent_rewards)) if recent_rewards else mean_reward

        pbar.set_postfix({
            "rwd":    f"{mean_reward:+.3f}",
            "ma20":   f"{reward_ma20:+.3f}",
            "KL":     f"{kl_val:.3f}",
            "loss":   f"{ppo_loss:.3f}",
            "len":    f"{avg_gen_len:.1f}",
            "nr":     f"{agg['number_recall']:.2f}",
            "halluc": f"{agg['hallucination']:.2f}",
        })

        log_entry = {
            "rl_epoch":     rl_epoch + 1,
            "step":         global_step,
            "mean_reward":  round(mean_reward, 4),
            "reward_ma20":  round(reward_ma20, 4),
            "kl":           round(kl_val, 4),
            "ppo_loss":     round(ppo_loss, 4),
            "avg_gen_len":  round(avg_gen_len, 2),
            **{k: round(v, 4) for k, v in agg.items()},
        }

        # Eval va save best
        if global_step > 0 and global_step % CONFIG["EVAL_EVERY_N_STEPS"] == 0:
            val_eval = quick_eval_on_val(n_samples=CONFIG["EVAL_VAL_SUBSET"])
            val_rougeL = val_eval["rougeL"]
            val_reward = val_eval["reward_mean"]
            log_entry["val_rougeL"] = val_rougeL
            log_entry["val_reward"] = val_reward

            # Composite: rougeL (0-100) + val_reward * 100 (~0-100).
            # Reward beam-search tren val phan anh chat luong eval, on dinh
            # hon reward sampled tu train batch.
            composite = (
                CONFIG["BEST_W_ROUGEL"] * val_rougeL
                + CONFIG["BEST_W_REWARD"] * 100.0 * val_reward
            )
            log_entry["composite"] = round(composite, 4)

            print(
                f"\nEpoch {rl_epoch+1}"
                f" | Step {global_step:4d}"
                f" | train_rwd={mean_reward:+.4f}"
                f" | ma20={reward_ma20:+.4f}"
                f" | KL={kl_val:.4f}"
                f" | val_rougeL={val_rougeL:.2f}"
                f" | val_reward={val_reward:+.4f}"
                f" | composite={composite:.2f}"
                f" | nr={agg['number_recall']:.2f}"
                f" | halluc={agg['hallucination']:.2f}"
            )

            if composite > best_score:
                best_score = composite
                best_meta = {
                    "step":        global_step,
                    "epoch":       rl_epoch + 1,
                    "val_rougeL":  val_rougeL,
                    "val_reward":  val_reward,
                    "composite":   composite,
                }
                BEST_DIR_S3 = save_actor(OUT_DIR / "best_model", step=global_step)
                best_meta["saved_dir"] = str(BEST_DIR_S3)
                with open(BEST_DIR_S3 / "best_meta.json", "w", encoding="utf-8") as f:
                    json.dump(best_meta, f, ensure_ascii=False, indent=2)
                print(
                    f">> Best model cap nhat "
                    f"(composite={composite:.2f}, val_rougeL={val_rougeL:.2f})"
                )

            # Sau eval, dop het VRAM cua beam-search graph
            clear_cuda()

        training_log.append(log_entry)
        global_step += 1

    print(f"\n--- Ket thuc RL Epoch {rl_epoch+1} ---")
    clear_cuda()

# Save log
with open(RESULTS_DIR / "ppo_training_log.json", "w", encoding="utf-8") as f:
    json.dump(training_log, f, ensure_ascii=False, indent=2)

# Neu chua tung save best (eval khong chay buoc nao) -> save state cuoi
if not (BEST_DIR_S3 / "config.json").exists():
    BEST_DIR_S3 = save_actor(OUT_DIR / "best_model", step=global_step)
    print("\nLuu checkpoint cuoi (chua co eval buoc nao chay).")

print("\n" + "=" * 80)
print(
    f"Training hoan tat."
    f"\nBest composite : {best_score:.2f}"
    f"\nBest meta      : {best_meta}"
    f"\nSkipped batches: {skipped_batches}"
    f"\nBest model dir : {BEST_DIR_S3}"
)
print("=" * 80)


Bat dau PPO Training - 3 RL epoch x 48 batch/epoch, batch=16, kl_penalty=abs, dtype=torch.bfloat16
--------------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_trainer.py:273: UserWarning: No dataset is provided. Make sure to set config.batch_size to the correct value before training.
  warnings.warn(


RL Epoch 1:   0%|          | 0/49 [00:00<?, ?it/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.



[INFO] PPO stats keys: ['objective/entropy', 'objective/kl', 'objective/kl_coef', 'objective/kl_dist', 'objective/logprobs', 'objective/ref_logprobs', 'ppo/learning_rate', 'ppo/loss/policy', 'ppo/loss/total', 'ppo/loss/value', 'ppo/mean_non_score_reward', 'ppo/mean_scores', 'ppo/policy/advantages', 'ppo/policy/advantages_mean', 'ppo/policy/approxkl', 'ppo/policy/clipfrac', 'ppo/policy/entropy', 'ppo/policy/policykl', 'ppo/policy/ratio', 'ppo/returns/mean', 'ppo/returns/var', 'ppo/std_scores', 'ppo/val/clipfrac', 'ppo/val/error', 'ppo/val/mean', 'ppo/val/var', 'ppo/val/var_explained', 'ppo/val/vpred', 'time/ppo/calc_stats', 'time/ppo/compute_advantages', 'time/ppo/compute_rewards', 'time/ppo/forward_pass', 'time/ppo/optimize_step', 'time/ppo/total', 'tokens/queries_dist', 'tokens/queries_len_mean', 'tokens/queries_len_std', 'tokens/responses_dist', 'tokens/responses_len_mean', 'tokens/responses_len_std']

[WARN] KL cao bat thuong step 4: 5.0443 (streak=1/3)

[WARN] KL cao bat thuong st

RL Epoch 2:   0%|          | 0/49 [00:00<?, ?it/s]


Epoch 2 | Step   50 | train_rwd=-0.6314 | ma20=-0.5350 | KL=4.9722 | val_rougeL=64.09 | val_reward=+0.5557 | composite=59.83 | nr=0.21 | halluc=0.71

[WARN] KL cao bat thuong step 69: 5.0505 (streak=1/3)

[WARN] KL cao bat thuong step 75: 5.2015 (streak=1/3)

Epoch 2 | Step   75 | train_rwd=-0.6030 | ma20=-0.5275 | KL=5.2015 | val_rougeL=64.41 | val_reward=+0.5597 | composite=60.19 | nr=0.26 | halluc=0.68

[WARN] KL cao bat thuong step 82: 5.0593 (streak=1/3)

[WARN] KL cao bat thuong step 85: 5.2582 (streak=1/3)

[WARN] KL cao bat thuong step 88: 5.0226 (streak=1/3)

[WARN] KL cao bat thuong step 89: 5.6329 (streak=2/3)

--- Ket thuc RL Epoch 2 ---


RL Epoch 3:   0%|          | 0/49 [00:00<?, ?it/s]


[WARN] KL cao bat thuong step 96: 5.0863 (streak=1/3)

[WARN] KL cao bat thuong step 97: 5.2339 (streak=2/3)

Epoch 3 | Step  100 | train_rwd=-0.3932 | ma20=-0.5583 | KL=4.3796 | val_rougeL=64.78 | val_reward=+0.5595 | composite=60.37 | nr=0.31 | halluc=0.65
>> Best model cap nhat (composite=60.37, val_rougeL=64.78)

[WARN] KL cao bat thuong step 105: 5.3950 (streak=1/3)

[WARN] KL cao bat thuong step 106: 5.3734 (streak=2/3)

[WARN] KL cao bat thuong step 110: 5.1820 (streak=1/3)

[WARN] KL cao bat thuong step 112: 5.6035 (streak=1/3)

[WARN] KL cao bat thuong step 116: 5.0276 (streak=1/3)

[WARN] KL cao bat thuong step 117: 5.3520 (streak=2/3)

[WARN] KL cao bat thuong step 121: 5.1739 (streak=1/3)

[WARN] KL cao bat thuong step 124: 5.4991 (streak=1/3)

Epoch 3 | Step  125 | train_rwd=-0.4989 | ma20=-0.5789 | KL=4.2782 | val_rougeL=65.90 | val_reward=+0.5626 | composite=61.08 | nr=0.23 | halluc=0.67
>> Best model cap nhat (composite=61.08, val_rougeL=65.90)

[WARN] KL cao bat thuon

In [ ]:
# Optional write test da duoc thay bang assert_writable_dir trong Cell 4.
print("Drive write test nam trong Cell 4.")

Drive write test nam trong Cell 4.


## 9. Validation Evaluation

Load best model Stage 3 và đánh giá đầy đủ trên validation set.


In [ ]:
# Cell 9 - Full validation evaluation.
# Free training-time model references before loading the saved base model for evaluation.

for _name in ("ppo_trainer", "actor_model", "ref_model"):
    if _name in dir():
        try:
            del globals()[_name]
        except KeyError:
            pass
clear_cuda()

eval_model     = AutoModelForSeq2SeqLM.from_pretrained(str(BEST_DIR_S3)).to(DEVICE)
eval_tokenizer = AutoTokenizer.from_pretrained(str(BEST_DIR_S3), use_fast=False)
eval_model.eval()


def length_compliance(src, pred):
    ratio = length_ratio_fn(src, pred)
    return float(CONFIG["MIN_RATIO"] <= ratio <= CONFIG["MAX_RATIO"])


def generate_summaries(model, tok, texts):
    outputs = []
    for i in range(0, len(texts), CONFIG["EVAL_BATCH"]):
        batch_texts = texts[i:i + CONFIG["EVAL_BATCH"]]
        enc = tok(
            batch_texts,
            max_length=CONFIG["MAX_INPUT_LEN"],
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(
                **enc,
                max_new_tokens=CONFIG["GEN_MAX_LEN"],
                min_new_tokens=CONFIG["GEN_MIN_LEN"],
                num_beams=CONFIG["NUM_BEAMS"],
                no_repeat_ngram_size=CONFIG["NO_REPEAT_NGRAM"],
                length_penalty=CONFIG["LENGTH_PENALTY"],
                early_stopping=True,
            )
        outputs.extend(tok.batch_decode(ids, skip_special_tokens=True))
    return outputs


def evaluate_predictions(srcs, golds, preds, tag="PPO"):
    rouge_result = rouge.compute(
        predictions=[normalize_for_rouge(x) for x in preds],
        references=[normalize_for_rouge(x) for x in golds],
        use_stemmer=False,
    )
    _, _, bert_f1 = bertscore(
        preds, golds,
        lang="vi",
        model_type="xlm-roberta-base",
        verbose=False,
    )
    nr = [number_recall(s, p, g)     for s, g, p in zip(srcs, golds, preds)]
    nh = [number_hallucination(s, p) for s, p in zip(srcs, preds)]
    lc = [length_compliance(s, p)    for s, p in zip(srcs, preds)]
    lr = [length_ratio_fn(s, p)      for s, p in zip(srcs, preds)]
    rw = [compute_reward(s, p, g)    for s, g, p in zip(srcs, golds, preds)]
    return {
        "tag":                  tag,
        "n_samples":            len(preds),
        "rouge1":               round(rouge_result["rouge1"]  * 100, 4),
        "rouge2":               round(rouge_result["rouge2"]  * 100, 4),
        "rougeL":               round(rouge_result["rougeL"]  * 100, 4),
        "bertscore_f1":         round(float(bert_f1.mean().item()) * 100, 4),
        "number_recall":        round(float(np.mean(nr)) * 100, 4),
        "number_hallucination": round(float(np.mean(nh)) * 100, 4),
        "length_compliance":    round(float(np.mean(lc)) * 100, 4),
        "length_ratio_mean":    round(float(np.mean(lr)), 4),
        "reward_mean":          round(float(np.mean(rw)), 4),
    }


val_src  = [r["source"]  for r in val_raw]
val_gold = [r["summary"] for r in val_raw]

val_preds   = generate_summaries(eval_model, eval_tokenizer, val_src)
metrics_val = evaluate_predictions(val_src, val_gold, val_preds, tag="Stage3_PPO_val")

print("Ket qua tren Val set:")
print(pd.DataFrame([metrics_val]).to_string(index=False))

with open(RESULTS_DIR / "eval_val_ppo.json", "w", encoding="utf-8") as f:
    json.dump(metrics_val, f, ensure_ascii=False, indent=2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Ket qua tren Val set:
           tag  n_samples  rouge1  rouge2  rougeL  bertscore_f1  number_recall  number_hallucination  length_compliance  length_ratio_mean  reward_mean
Stage3_PPO_val         94 79.0821 66.9838  65.515       93.0955        64.6363                 2.838            38.2979             0.1869        0.572


## 10. Final Test Evaluation and Export

Chỉ chạy một lần sau khi đã chốt model. Cell này đánh giá test set và export toàn bộ kết quả phục vụ báo cáo.


In [ ]:
# Cell 10 - Final test evaluation and export.
# Cell nay co the chay doc lap, khong can chay cac cell truoc.
#
# Yeu cau duy nhat: Google Drive da duoc mount.
# Cell tu load: model, tokenizer, dataset, roi danh gia tren test set.
#
# DUONG DAN CAN KIEM TRA TRUOC KHI CHAY:
#   STANDALONE_BEST_DIR   — best_model Stage 3 duoc luu boi Cell 8
#                           Cau truc: .../ViT5_stage_3/runs/ppo_run/best_model
#   STANDALONE_DATA_ROOT  — thu muc chua test/dataset.jsonl
#   STANDALONE_RESULTS_DIR — noi luu ket qua

import os
import re
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from bert_score import score as bertscore
import evaluate

# ------------------------------------------------------------------
# DUONG DAN — chinh lai neu sai
# ------------------------------------------------------------------

# Best model Stage 3: Cell 8 luu tai OUT_DIR / "best_model"
# OUT_DIR = BASE_DIR_S3 / "runs" / "ppo_run"
STANDALONE_BEST_DIR = (
    "/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP"
    "/ViT5_stage_3/runs/ppo_run/best_model"
)

# Thu muc data goc — cung voi DATA_ROOT trong Cell 3
STANDALONE_DATA_ROOT = (
    "/content/drive/MyDrive/MyNLPData/data"
)

# Noi luu output — tu dong tao neu chua co
STANDALONE_RESULTS_DIR = (
    "/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP"
    "/ViT5_stage_3/runs/ppo_run/results"
)

# ------------------------------------------------------------------
# Generation config — giu nguyen voi Cell 8 de ket qua nhat quan
# ------------------------------------------------------------------
EVAL_CONFIG = {
    "MAX_INPUT_LEN":  1024,
    "GEN_MAX_LEN":     320,
    "GEN_MIN_LEN":      64,
    "NUM_BEAMS":          4,
    "NO_REPEAT_NGRAM":    3,
    "LENGTH_PENALTY":   1.0,
    "EVAL_BATCH":         8,
    "MIN_RATIO":        0.18,
    "MAX_RATIO":        0.27,
    "TARGET_RATIO":    0.225,
    "W_NUMBER":         0.45,
    "W_LENGTH":         0.20,
    "W_HALLUCINATION":  0.20,
    "W_DIVERSITY":      0.15,
    "HALLUC_HARD_THRESHOLD": 0.5,
    "HALLUC_HARD_PENALTY":   1.0,
    "NUMBER_USE_GOLD_FOR_REWARD": True,
}

# ------------------------------------------------------------------
# Setup device
# ------------------------------------------------------------------
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())

# Mount Drive neu chua mount
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

# Kiem tra duong dan ton tai
best_dir    = Path(STANDALONE_BEST_DIR)
data_root   = Path(STANDALONE_DATA_ROOT)
results_dir = Path(STANDALONE_RESULTS_DIR)

if not best_dir.exists():
    raise FileNotFoundError(
        f"Khong tim thay best model tai: {best_dir}\n"
        "Kiem tra lai STANDALONE_BEST_DIR."
    )
test_path = data_root / "test" / "dataset.jsonl"
if not test_path.exists():
    raise FileNotFoundError(
        f"Khong tim thay test data tai: {test_path}\n"
        "Kiem tra lai STANDALONE_DATA_ROOT."
    )
results_dir.mkdir(parents=True, exist_ok=True)

print("Best model  :", best_dir)
print("Test data   :", test_path)
print("Results dir :", results_dir)
print("Device      :", DEVICE)
print("Precision   :", "bf16" if USE_BF16 else "fp32")

# ------------------------------------------------------------------
# Load model va tokenizer tu STANDALONE_BEST_DIR
# ------------------------------------------------------------------
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(str(best_dir))
print(f"Tokenizer: {type(tokenizer).__name__}, vocab={tokenizer.vocab_size}")

print("Loading model...")
eval_model = AutoModelForSeq2SeqLM.from_pretrained(
    str(best_dir),
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
).to(DEVICE)
eval_model.eval()
print(f"Model: {sum(p.numel() for p in eval_model.parameters())/1e6:.1f}M params")

rouge = evaluate.load("rouge")

# ------------------------------------------------------------------
# Load test dataset
# ------------------------------------------------------------------
NUM_RE = re.compile(r"\d+(?:[.,]\d+)*")

def extract_numbers(text):
    return set(NUM_RE.findall(text or ""))

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

test_raw = []
for row in load_jsonl(test_path):
    src = str(row.get("source", "")).strip()
    tgt = str(row.get("summary", "")).strip()
    if src and tgt:
        test_raw.append({"id": row.get("id", ""), "source": src, "summary": tgt})

test_src  = [r["source"]  for r in test_raw]
test_gold = [r["summary"] for r in test_raw]
test_ids  = [r["id"]      for r in test_raw]
print(f"\nTest set: {len(test_raw)} mau")

# ------------------------------------------------------------------
# Reward functions (tu Cell 7, tu chua — khong phu thuoc bien ngoai)
# ------------------------------------------------------------------
def number_recall(src, pred, target_text=None):
    target_nums = (
        extract_numbers(target_text) if target_text is not None
        else extract_numbers(src)
    )
    if not target_nums:
        # Khong co so can giu -> khong the sai -> reward day du
        return 1.0
    return len(target_nums & extract_numbers(pred)) / len(target_nums)

def number_hallucination(src, pred, gold=None):
    pred_nums = extract_numbers(pred)
    if not pred_nums:
        return 0.0
    allowed = extract_numbers(src)
    if gold is not None and EVAL_CONFIG["NUMBER_USE_GOLD_FOR_REWARD"]:
        allowed = allowed | extract_numbers(gold)
    return len(pred_nums - allowed) / len(pred_nums)

def length_ratio_fn(src, pred):
    return len(str(pred).split()) / max(len(str(src).split()), 1)

def length_score(src, pred):
    ratio = length_ratio_fn(src, pred)
    if EVAL_CONFIG["MIN_RATIO"] <= ratio <= EVAL_CONFIG["MAX_RATIO"]:
        return 1.0
    return max(0.0, 1.0 - abs(ratio - EVAL_CONFIG["TARGET_RATIO"]) / EVAL_CONFIG["TARGET_RATIO"])

def diversity_score(pred):
    words = str(pred).split()
    if not words:
        return 0.0
    return len(set(words)) / len(words)

def length_compliance(src, pred):
    return float(EVAL_CONFIG["MIN_RATIO"] <= length_ratio_fn(src, pred) <= EVAL_CONFIG["MAX_RATIO"])

def compute_reward_breakdown(src, pred, gold=None):
    target = gold if EVAL_CONFIG["NUMBER_USE_GOLD_FOR_REWARD"] else None
    nr = number_recall(src, pred, target)
    nh = number_hallucination(src, pred, gold)
    ls = length_score(src, pred)
    ds = diversity_score(pred)
    raw = (
        EVAL_CONFIG["W_NUMBER"]          * nr
        + EVAL_CONFIG["W_LENGTH"]        * ls
        + EVAL_CONFIG["W_DIVERSITY"]     * ds
        - EVAL_CONFIG["W_HALLUCINATION"] * nh
    )
    hard = 0.0
    if nh > EVAL_CONFIG["HALLUC_HARD_THRESHOLD"]:
        hard = -EVAL_CONFIG["HALLUC_HARD_PENALTY"]
        raw += hard
    return {
        "reward":        float(np.clip(raw, -1.0, 1.0)),
        "number_recall": float(nr),
        "hallucination": float(nh),
        "length_score":  float(ls),
        "diversity":     float(ds),
        "hard_penalty":  float(hard),
    }

# ------------------------------------------------------------------
# Generate summaries tren test set
# ------------------------------------------------------------------
def normalize_for_rouge(text):
    return "\n".join(str(text).strip().split())

def generate_summaries(texts):
    outputs = []
    for i in range(0, len(texts), EVAL_CONFIG["EVAL_BATCH"]):
        batch = texts[i:i + EVAL_CONFIG["EVAL_BATCH"]]
        enc = tokenizer(
            batch,
            max_length=EVAL_CONFIG["MAX_INPUT_LEN"],
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            ids = eval_model.generate(
                **enc,
                max_new_tokens=EVAL_CONFIG["GEN_MAX_LEN"],
                min_new_tokens=EVAL_CONFIG["GEN_MIN_LEN"],
                num_beams=EVAL_CONFIG["NUM_BEAMS"],
                no_repeat_ngram_size=EVAL_CONFIG["NO_REPEAT_NGRAM"],
                length_penalty=EVAL_CONFIG["LENGTH_PENALTY"],
                early_stopping=True,
            )
        outputs.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return outputs

print("\nGenerating test predictions...")
test_preds = generate_summaries(test_src)
print(f"Done: {len(test_preds)} predictions")

# ------------------------------------------------------------------
# Tinh metrics
# ------------------------------------------------------------------
print("\nTinh ROUGE...")
rouge_result = rouge.compute(
    predictions=[normalize_for_rouge(x) for x in test_preds],
    references=[normalize_for_rouge(x) for x in test_gold],
    use_stemmer=False,
)

print("Tinh BERTScore (co the mat 1-2 phut)...")
_, _, bert_f1 = bertscore(
    test_preds, test_gold,
    lang="vi",
    model_type="xlm-roberta-base",
    verbose=False,
)

# Tinh reward tung record
records = []
for i in range(len(test_src)):
    bd = compute_reward_breakdown(test_src[i], test_preds[i], test_gold[i])
    records.append({
        "id":                   test_ids[i],
        "source":               test_src[i],
        "gold":                 test_gold[i],
        "pred_ppo":             test_preds[i],
        "number_recall":        bd["number_recall"],
        "number_hallucination": bd["hallucination"],
        "length_score":         bd["length_score"],
        "diversity":            bd["diversity"],
        "length_ratio":         length_ratio_fn(test_src[i], test_preds[i]),
        "length_compliance":    length_compliance(test_src[i], test_preds[i]),
        "reward":               bd["reward"],
    })

metrics_test = {
    "tag":                  "Stage3_PPO_test",
    "n_samples":            len(test_preds),
    "rouge1":               round(rouge_result["rouge1"]  * 100, 4),
    "rouge2":               round(rouge_result["rouge2"]  * 100, 4),
    "rougeL":               round(rouge_result["rougeL"]  * 100, 4),
    "bertscore_f1":         round(float(bert_f1.mean().item()) * 100, 4),
    "number_recall":        round(float(np.mean([r["number_recall"]        for r in records])) * 100, 4),
    "number_hallucination": round(float(np.mean([r["number_hallucination"] for r in records])) * 100, 4),
    "length_compliance":    round(float(np.mean([r["length_compliance"]    for r in records])) * 100, 4),
    "length_ratio_mean":    round(float(np.mean([r["length_ratio"]         for r in records])), 4),
    "reward_mean":          round(float(np.mean([r["reward"]               for r in records])), 4),
}

print("\n" + "=" * 65)
print("KET QUA TREN TEST SET — Stage 3 PPO")
print("=" * 65)
print(pd.DataFrame([metrics_test]).T.to_string(header=False))
print("=" * 65)

# ------------------------------------------------------------------
# Export
# ------------------------------------------------------------------
# Load best_meta neu ton tai de ghi vao ket qua
best_meta = {}
meta_path = best_dir / "best_meta.json"
if meta_path.exists():
    with open(meta_path, encoding="utf-8") as f:
        best_meta = json.load(f)

final_results = {
    "model":       "ViT5-base Stage3 PPO",
    "best_dir":    str(best_dir),
    "data_root":   str(data_root),
    "test_size":   len(test_raw),
    "eval_config": EVAL_CONFIG,
    "best_meta":   best_meta,
    "metrics":     metrics_test,
    "records":     records,
}

with open(results_dir / "final_results_stage3.json", "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

with open(results_dir / "stage3_predictions.jsonl", "w", encoding="utf-8") as f:
    for row in records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

pd.DataFrame([metrics_test]).to_csv(
    results_dir / "stage3_metrics.csv", index=False
)

print("\nFiles da luu:")
print(" -", results_dir / "final_results_stage3.json")
print(" -", results_dir / "stage3_predictions.jsonl")
print(" -", results_dir / "stage3_metrics.csv")
print("Best model dir:", best_dir)


Best model  : /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/best_model
Test data   : /content/drive/MyDrive/MyNLPData/data/test/dataset.jsonl
Results dir : /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/results
Device      : cuda
Precision   : bf16

Loading tokenizer...


You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Tokenizer: T5TokenizerFast, vocab=36096
Loading model...
Model: 226.0M params


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



Test set: 105 mau

Generating test predictions...
Done: 105 predictions

Tinh ROUGE...
Tinh BERTScore (co the mat 1-2 phut)...


...

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]


KET QUA TREN TEST SET — Stage 3 PPO
tag                   Stage3_PPO_test
n_samples                         105
rouge1                        77.0427
rouge2                        63.5902
rougeL                        62.2986
bertscore_f1                  92.2244
number_recall                 65.0596
number_hallucination           2.0299
length_compliance             43.8095
length_ratio_mean              0.1821
reward_mean                    0.5755

Files da luu:
 - /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/results/final_results_stage3.json
 - /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/results/stage3_predictions.jsonl
 - /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/results/stage3_metrics.csv
Best model dir: /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/best_model


In [ ]:
import json
from pathlib import Path

best_dir = Path(
    "/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP"
    "/ViT5_stage_3/runs/ppo_run/best_model"
)

# Kiem tra best_meta.json — neu PPO co luu duoc thi file nay ton tai
meta_path = best_dir / "best_meta.json"
if meta_path.exists():
    with open(meta_path) as f:
        print("best_meta:", json.load(f))
else:
    print("KHONG CO best_meta.json -> PPO chua luu duoc best model")

# Kiem tra thoi gian tao file
config_path = best_dir / "config.json"
if config_path.exists():
    import os, datetime
    mtime = os.path.getmtime(config_path)
    print("config.json modified:", datetime.datetime.fromtimestamp(mtime))
else:
    print("KHONG CO config.json")

# Kiem tra repaired dir
repaired = Path(
    "/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP"
    "/ViT5_stage_3/stage2_repaired_for_stage3"
)
print("\nFiles in repaired dir:", list(repaired.iterdir()) if repaired.exists() else "KHONG TON TAI")
print("Files in best_model dir:", list(best_dir.iterdir()) if best_dir.exists() else "KHONG TON TAI")

best_meta: {'step': 125, 'epoch': 3, 'val_rougeL': 65.9029, 'val_reward': 0.5626, 'composite': 61.081450000000004, 'saved_dir': '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/best_model'}
config.json modified: 2026-05-10 12:17:36

Files in repaired dir: KHONG TON TAI
Files in best_model dir: [PosixPath('/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/best_model/config.json'), PosixPath('/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/best_model/generation_config.json'), PosixPath('/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/best_model/model.safetensors'), PosixPath('/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/best_model/added_tokens.json'), PosixPath('/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/best_model/spiece.model'), PosixPath('/content/drive/MyDrive/C.Sci